# YOLO11m 베이스라인

노션 칸반 '진행 중' 항목 중 YOLO11m 베이스라인 기준을 따릅니다. 저장소의 `.py` 코드는 쓰지 않고 이 노트북 안에서 모두 작성합니다.

1. Colab 연결 (GPU 확인, 저장소·패키지 준비)
2. 설정
3. 경로와 장치
4. 데이터 준비 (원본 JSON → 라벨 보정 → combo 5 fold 분할 → YOLO 형식)
5. 학습 (MuSGD, early stopping 5, best.pt는 mAP75-95 기준, epoch별 시간 tqdm)
6. 평가 (mAP75-95 · Precision · Recall · 추론속도)
7. 혼동행렬 conf 0.001 vs 0.25
8. 실패 사례 시각화

**실행 순서**
- 로컬: `SMOKE = True`로 위에서부터 실행해 동작만 확인합니다 (1 epoch, 학습 사진 10%).
- Colab: 커널을 Colab GPU 서버로 고르고, `SMOKE = False`로 바꾼 뒤 위에서부터 실행합니다.

## 1. Colab 연결

GPU는 코드가 아니라 **커널 선택**으로 받습니다. VS Code 노트북 오른쪽 위 커널 선택 → **Colab** → 새 서버 → **GPU**(T4 등).

Colab이면 이 셀이 아래를 준비하고, 로컬에서는 아무것도 하지 않습니다.

- GPU 확인 (없으면 멈춤)
- 저장소 받기: 보정 기록 csv(`data/processed/*.csv`)와 폴더 구조를 쓰려고 받습니다. 저장소의 `.py`는 import하지 않습니다.
- ultralytics 설치: 로컬 `uv.lock`과 같은 8.4.182

결과는 노트북 출력으로 남깁니다. Colab 서버의 파일(best.pt 등)은 런타임이 끝나면 사라집니다.

In [ ]:
import os
from pathlib import Path

# VS Code에서 Colab 커널에 붙으면 COLAB_RELEASE_TAG가 없을 수 있어 /content 폴더도 본다
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()

if IN_COLAB:
    import torch

    assert torch.cuda.is_available(), "GPU가 아닙니다. 커널을 Colab GPU 서버로 다시 고르세요."
    print(torch.cuda.get_device_name(0))

    REPO_DIR = Path("/content/pill-bbox")
    if not REPO_DIR.exists():
        !git clone -q -b main https://github.com/def-uk/pill-bbox.git {REPO_DIR}
    os.chdir(REPO_DIR)

    %pip install -q ultralytics==8.4.182

print(f"Colab: {IN_COLAB}  /  작업 폴더: {Path.cwd()}")

## 2. 설정

모든 모델 베이스라인에 같게 적용하는 값입니다. 바꾸지 않습니다.

In [ ]:
# ===== 베이스라인 공통 설정 (노션 1번 항목) =====
MODEL = "yolo11m.pt"  # COCO 사전학습 가중치에서 시작하는 전체 파인튜닝 (freeze 없음)
SEED = 42
EPOCHS = 30
PATIENCE = 5  # 검증 mAP75-95가 5 epoch 동안 오르지 않으면 학습을 멈춘다
BATCH = 16
IMGSZ = 640
OPTIMIZER = "MuSGD"
LR0 = 0.001
EVAL_CONF = 0.001  # 평가(mAP 계산)에 쓰는 신뢰도 하한
PLOT_CONF = 0.25  # 비교 그림에 박스를 그릴 때 쓰는 신뢰도 하한. 평가용과 구분한다
NMS_IOU = 0.5  # YOLO11은 NMS 후처리를 쓰므로 이 값이 실제로 적용된다
MAX_DET = 4  # 사진 한 장에서 남기는 최대 박스 수 (데이터의 사진당 알약은 3–4개)

# ===== 분할 (노션 2번 항목) =====
# 알약 조합 단위로 나눈다. 같은 조합을 다른 각도로 찍은 사진이 train·val에 갈라지지 않게 한다
SPLIT_UNIT = "combo"
N_FOLDS = 5  # 조합을 5묶음으로 나눠 fold마다 한 묶음(약 20%)을 val로 쓴다
FOLD = 0
RARE_MAX_BOXES = 5  # 박스가 이 수 이하인 약이 들어간 조합은 val에 넣지 않고 train에 고정한다

# ===== 실행 =====
SMOKE = True  # True: 1 epoch, 학습 사진 10%로 동작만 확인. Colab GPU 본 학습은 False

## 3. 경로와 장치

원본 데이터를 읽는 곳과 이 노트북이 결과를 쓰는 곳입니다. 결과 폴더는 모두 Git에서 제외되는 위치(`data/processed/`, `runs/`)에 둡니다.

In [ ]:
import random

import numpy as np
import pandas as pd
import torch

# 노트북이 어느 폴더에서 열려도 pyproject.toml이 있는 곳을 프로젝트 루트로 쓴다
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()
)
# ultralytics는 yolo11m.pt 등 가중치를 현재 폴더에 받으므로 프로젝트 루트에서 실행한다
os.chdir(PROJECT_ROOT)
NAME = "yolo11m_kwon"

# 읽기: 원본 (고치지 않는다)과 보정 기록
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"
CORRECTION_DIR = PROJECT_ROOT / "data" / "processed"

# 쓰기: 데이터 준비 결과 (라벨 표, 분할 목록, YOLO 형식 이미지·라벨, data.yaml)
DATA_OUT_DIR = PROJECT_ROOT / "data" / "processed" / NAME
YOLO_DIR = DATA_OUT_DIR / f"yolo_{SPLIT_UNIT}_fold{FOLD}"
DATA_YAML = YOLO_DIR / "data.yaml"

# 쓰기: 학습·평가 결과 (가중치 best.pt·last.pt, results.csv, 그림)
RUNS_OUT_DIR = PROJECT_ROOT / "runs" / NAME
EXP = f"{'smoke_' if SMOKE else ''}{Path(MODEL).stem}_{OPTIMIZER}_lr{LR0:g}_{SPLIT_UNIT}_fold{FOLD}"
EXP_DIR = RUNS_OUT_DIR / EXP
WEIGHTS_DIR = EXP_DIR / "weights"


def get_device() -> str:
    """CUDA(Windows/Linux) > MPS(Mac) > CPU 순으로 쓸 수 있는 장치를 고른다."""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def set_seed(seed: int) -> None:
    """분할·학습이 다시 실행해도 같게 나오도록 난수를 고정한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


DEVICE = get_device()
set_seed(SEED)
if not SMOKE and DEVICE != "cuda":
    raise RuntimeError(f"본 학습은 GPU에서 합니다(지금 {DEVICE}). 동작 확인은 SMOKE = True.")

print(f"장치: {DEVICE}  /  실험 이름: {EXP}")
pd.DataFrame(
    [
        ("읽기", "원본 train 이미지", TRAIN_IMG_DIR),
        ("읽기", "원본 train JSON", TRAIN_ANN_DIR),
        ("읽기", "원본 test 이미지", TEST_IMG_DIR),
        ("읽기", "보정 기록 csv", CORRECTION_DIR),
        ("쓰기", "데이터 준비 결과", DATA_OUT_DIR),
        ("쓰기", "YOLO 데이터셋", YOLO_DIR),
        ("쓰기", "data.yaml", DATA_YAML),
        ("쓰기", "학습·평가 결과", EXP_DIR),
        ("쓰기", "best.pt / last.pt", WEIGHTS_DIR),
    ],
    columns=["구분", "내용", "경로"],
).assign(있음=lambda df: df["경로"].map(Path.exists))

In [ ]:
def show_outputs(root: Path, max_files: int = 10) -> pd.DataFrame:
    """결과 폴더 아래 파일을 폴더별로 몇 개씩 보여 준다. 저장이 제대로 됐는지 확인용."""
    if not root.exists():
        print(f"아직 없습니다: {root}")
        return pd.DataFrame()
    rows = []
    for folder in sorted({p.parent for p in root.rglob("*") if p.is_file()}):
        files = sorted(p for p in folder.iterdir() if p.is_file())
        name = str(folder.relative_to(root))
        for p in files[:max_files]:
            rows.append((name, p.name, f"{p.stat().st_size / 1024:.1f} KB"))
        if len(files) > max_files:
            rows.append((name, f"... 외 {len(files) - max_files}개", ""))
    print(f"{root}")
    return pd.DataFrame(rows, columns=["폴더", "파일", "크기"])


# 처음에는 비어 있다. 데이터 준비·학습 뒤에 다시 실행해 결과를 확인한다
show_outputs(DATA_OUT_DIR)

## 4. 데이터 준비

### 4-1. 원본 확인

`data/raw`에 train 사진과 JSON이 있는지 봅니다. 없으면 여기서 멈춥니다.

In [ ]:
if not (TRAIN_IMG_DIR.is_dir() and TRAIN_ANN_DIR.is_dir()):
    raise RuntimeError(f"원본 데이터가 없습니다: {RAW_DIR}")

print(f"train 사진 {len(list(TRAIN_IMG_DIR.glob('*.png')))}장")
print(f"train JSON {len(list(TRAIN_ANN_DIR.rglob('*.json')))}개")

### 4-2. 원본 JSON 읽기

JSON 하나에 사진 하나·알약 하나의 정보가 있습니다. 박스 한 개를 한 행으로 모읍니다. bbox는 원본 픽셀 기준 `[왼쪽 위 x, 왼쪽 위 y, 너비, 높이]`입니다.

In [ ]:
import json

BOX = ["x", "y", "w", "h"]

rows = []
for path in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    ann = json.loads(path.read_text(encoding="utf-8"))
    img = ann["images"][0]
    for a in ann["annotations"]:
        rows.append(
            {
                "file_name": img["file_name"],
                "width": img["width"],
                "height": img["height"],
                "category_id": a["category_id"],
                "dl_name": img["dl_name"],
                **dict(zip(BOX, a["bbox"], strict=True)),
            }
        )
raw_labels = pd.DataFrame(rows)

n_images = len(list(TRAIN_IMG_DIR.glob("*.png")))
print(f"박스 {len(raw_labels)}개, 클래스 {raw_labels['category_id'].nunique()}개")
print(f"라벨이 있는 사진 {raw_labels['file_name'].nunique()}장 / 원본 사진 {n_images}장")
raw_labels.head()

### 4-3. 라벨 보정 반영

원본 JSON은 고치지 않고, 보정 기록 csv를 읽어 라벨 표에만 반영합니다.

- `fixed_annotations.csv`: 위치가 틀린 박스의 수정 전·후 좌표 (5개)
- `added_annotations.csv`: 라벨이 빠진 알약 (8개)

반영한 뒤 두 가지를 검사합니다. 파일 이름의 조합(`K-코드1-코드2-...`)에 적힌 알약 수와 박스 수가 같은지, 박스가 사진 안에 있는지 봅니다.

In [ ]:
fixed = pd.read_csv(CORRECTION_DIR / "fixed_annotations.csv")
added = pd.read_csv(CORRECTION_DIR / "added_annotations.csv")
labels = raw_labels.copy()

# 위치 수정: 파일·클래스·수정 전 좌표가 모두 같은 박스를 찾아 새 좌표로 바꾼다
for fix in fixed.itertuples():
    hit = (
        (labels["file_name"] == fix.file_name)
        & (labels["category_id"] == fix.category_id)
        & (labels[BOX] == [fix.old_x, fix.old_y, fix.old_w, fix.old_h]).all(axis=1)
    )
    if hit.sum() != 1:
        raise ValueError(f"수정할 박스를 하나로 찾지 못했습니다: {fix.file_name} {fix.category_id}")
    labels.loc[hit, BOX] = [fix.x, fix.y, fix.w, fix.h]

# 누락 추가: 사진 크기는 같은 사진의 다른 박스에서, 약 이름은 같은 클래스의 다른 박스에서 가져온다
size = labels.drop_duplicates("file_name").set_index("file_name")[["width", "height"]]
names = labels.drop_duplicates("category_id").set_index("category_id")["dl_name"]
new = added[["file_name", "category_id", *BOX]].join(size, on="file_name")
new["dl_name"] = new["category_id"].map(names)
if new.isna().any().any():
    missing = new[new.isna().any(axis=1)]
    raise ValueError(f"추가할 박스의 사진 크기나 약 이름을 찾지 못했습니다:\n{missing}")
labels = pd.concat([labels, new[labels.columns]], ignore_index=True)
labels = labels.sort_values(["file_name", "category_id"], ignore_index=True)
labels["combo"] = labels["file_name"].str.split("_").str[0]

# 검사
n_pills = labels.groupby("file_name")["combo"].first().str.count("-")  # K-a-b-c → 3개
n_boxes = labels.groupby("file_name").size()
outside = (
    (labels["x"] < 0)
    | (labels["y"] < 0)
    | (labels["x"] + labels["w"] > labels["width"])
    | (labels["y"] + labels["h"] > labels["height"])
    | (labels["w"] <= 0)
    | (labels["h"] <= 0)
)
print(f"박스 {len(raw_labels)}개 → 위치 수정 {len(fixed)}개, 추가 {len(new)}개 → {len(labels)}개")
print(f"알약 수와 박스 수가 다른 사진: {(n_pills != n_boxes).sum()}장 (0이어야 함)")
print(f"사진 밖으로 나간 박스: {outside.sum()}개 (0이어야 함)")

DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)
labels.to_csv(DATA_OUT_DIR / "labels.csv", index=False, encoding="utf-8")
labels.head()

### 4-4. 클래스 표

`category_id`(원본 약 번호)를 0부터 시작하는 YOLO 클래스 번호(`yolo_idx`)로 바꿉니다. `rare`는 박스가 5개 이하인 약입니다.

In [ ]:
class_table = (
    labels.groupby("category_id")
    .agg(
        dl_name=("dl_name", "first"),
        n_boxes=("file_name", "size"),
        n_images=("file_name", "nunique"),
        n_combos=("combo", "nunique"),
    )
    .reset_index()
)
class_table.insert(0, "yolo_idx", range(len(class_table)))
class_table["rare"] = class_table["n_boxes"] <= RARE_MAX_BOXES
class_table.to_csv(DATA_OUT_DIR / "classes.csv", index=False, encoding="utf-8")

n_rare = class_table["rare"].sum()
print(f"클래스 {len(class_table)}개, 그중 박스 {RARE_MAX_BOXES}개 이하 {n_rare}개")
class_table.sort_values("n_boxes").head(20)

### 4-5. combo 5 fold 분할

1. 희귀 약(박스 5개 이하)이 들어간 조합은 train에 고정합니다(`fold = -1`). val에만 들어가면 학습 때 한 번도 못 보기 때문입니다.
2. 나머지 조합을 seed로 섞은 뒤, 사진이 많은 조합부터 사진 수가 가장 적은 fold에 넣습니다. 그래서 fold마다 val 사진 수가 고르게 됩니다.
3. fold k는 k번 묶음을 val, 나머지를 train으로 씁니다.

In [ ]:
rare_ids = class_table.loc[class_table["rare"], "category_id"]
pinned = set(labels.loc[labels["category_id"].isin(rare_ids), "combo"])
combo_images = labels.groupby("combo")["file_name"].nunique()
pool = combo_images.drop(list(pinned)).sample(frac=1, random_state=SEED)

fold_of = {}
fold_images = [0] * N_FOLDS
# stable 정렬이라 사진 수가 같은 조합끼리는 섞은 순서가 유지된다
for combo, n in pool.sort_values(ascending=False, kind="stable").items():
    k = int(np.argmin(fold_images))
    fold_of[combo] = k
    fold_images[k] += n

images = labels.groupby("file_name", as_index=False)["combo"].first()
images["fold"] = images["combo"].map(fold_of).fillna(-1).astype(int)  # -1: train 고정
images.to_csv(DATA_OUT_DIR / f"split_{SPLIT_UNIT}_s{SEED}.csv", index=False, encoding="utf-8")

rows = []
for k in range(N_FOLDS):
    is_val = labels["file_name"].isin(images.loc[images["fold"] == k, "file_name"])
    val_cls = set(labels.loc[is_val, "category_id"])
    train_cls = set(labels.loc[~is_val, "category_id"])
    rows.append(
        {
            "fold": k,
            "val 조합": labels.loc[is_val, "combo"].nunique(),
            "val 사진": labels.loc[is_val, "file_name"].nunique(),
            "val 사진 비율": labels.loc[is_val, "file_name"].nunique() / len(images),
            "val 박스": is_val.sum(),
            "val 클래스": len(val_cls),
            "val에만 있는 클래스": len(val_cls - train_cls),
        }
    )
print(f"조합 {len(combo_images)}개 = train 고정 {len(pinned)}개 + 5 fold로 나눈 {len(pool)}개")
pd.DataFrame(rows).style.format({"val 사진 비율": "{:.1%}"})

### 4-6. YOLO 형식으로 변환

`FOLD`의 train/val로 아래 구조를 만듭니다. 이미지는 복사하지 않고 원본을 가리키는 링크만 만듭니다. 이전 결과가 있으면 지우고 새로 만듭니다.

```text
yolo_combo_fold0/
├── data.yaml
├── images/{train,val}/*.png   (원본 링크)
└── labels/{train,val}/*.txt   (클래스 번호, 중심 x, 중심 y, 너비, 높이 — 사진 크기로 나눈 0–1 값)
```

In [ ]:
import shutil

import yaml

# 이전 분할·fold의 파일과 ultralytics 라벨 캐시가 섞이지 않게 새로 만든다
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

to_idx = dict(zip(class_table["category_id"], class_table["yolo_idx"], strict=True))
split_of = images.set_index("file_name")["fold"].map(lambda k: "val" if k == FOLD else "train")

for file_name, boxes in labels.groupby("file_name"):
    split = split_of[file_name]
    img_link = YOLO_DIR / "images" / split / file_name
    txt_path = YOLO_DIR / "labels" / split / f"{Path(file_name).stem}.txt"
    img_link.parent.mkdir(parents=True, exist_ok=True)
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    img_link.symlink_to(TRAIN_IMG_DIR / file_name)
    lines = [
        f"{to_idx[b.category_id]} {(b.x + b.w / 2) / b.width:.6f} {(b.y + b.h / 2) / b.height:.6f} "
        f"{b.w / b.width:.6f} {b.h / b.height:.6f}"
        for b in boxes.itertuples()
    ]
    txt_path.write_text("\n".join(lines) + "\n")

# names는 원본 약 번호(category_id). 한글 약 이름은 classes.csv에서 찾는다
data_cfg = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {int(i): str(c) for c, i in to_idx.items()},
}
DATA_YAML.write_text(yaml.safe_dump(data_cfg, allow_unicode=True, sort_keys=False))
print(DATA_YAML.read_text()[:300])

### 4-7. 변환 검사

- 사진 수와 라벨 txt 수가 split마다 같은지
- txt를 다시 픽셀 좌표로 되돌렸을 때 라벨 표와 같은지 (반올림 오차 1픽셀 미만)
- val 사진 몇 장에 txt의 박스를 그려 눈으로 확인

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

back = []
for split in ["train", "val"]:
    imgs = sorted((YOLO_DIR / "images" / split).glob("*.png"))
    txts = sorted((YOLO_DIR / "labels" / split).glob("*.txt"))
    n_broken = sum(not p.exists() for p in imgs)
    print(f"{split}: 사진 {len(imgs)}장, 라벨 {len(txts)}개, 깨진 링크 {n_broken}개")
    for txt in txts:
        for line in txt.read_text().splitlines():
            cls, cx, cy, w, h = line.split()
            back.append((f"{txt.stem}.png", int(cls), float(cx), float(cy), float(w), float(h)))

back = pd.DataFrame(back, columns=["file_name", "yolo_idx", "cx", "cy", "bw", "bh"])
back = back.join(size, on="file_name")
back["category_id"] = back["yolo_idx"].map(class_table.set_index("yolo_idx")["category_id"])
back["x"] = (back["cx"] - back["bw"] / 2) * back["width"]
back["y"] = (back["cy"] - back["bh"] / 2) * back["height"]
back["w"] = back["bw"] * back["width"]
back["h"] = back["bh"] * back["height"]
keys = ["file_name", "category_id"]
merged = labels.merge(back, on=keys, suffixes=("", "_back"))
max_err = max((merged[c] - merged[f"{c}_back"]).abs().max() for c in BOX)
print(f"박스 수: 라벨 표 {len(labels)}개, txt {len(back)}개, 짝지은 박스 {len(merged)}개")
print(f"픽셀 좌표 최대 오차: {max_err:.3f} (1 미만이어야 함)")

# val 사진 4장에 txt의 박스를 그린다
samples = back.loc[back["file_name"].map(split_of) == "val", "file_name"].drop_duplicates()
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, file_name in zip(axes, samples.sample(4, random_state=SEED), strict=True):
    img = Image.open(TRAIN_IMG_DIR / file_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    for b in back[back["file_name"] == file_name].itertuples():
        draw.rectangle([b.x, b.y, b.x + b.w, b.y + b.h], outline="lime", width=6)
        draw.text((b.x + 8, b.y + 8), str(b.category_id), fill="lime", font_size=40)
    ax.imshow(img)
    ax.set_title(file_name.split("_")[0], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 데이터 준비 결과가 저장된 곳
show_outputs(DATA_OUT_DIR, max_files=5)

## 5. 학습

### 5-1. best.pt 기준을 mAP75-95로

ultralytics는 매 epoch 검증 뒤 fitness 값이 가장 높은 epoch를 `best.pt`로 저장하고, fitness가 `PATIENCE` epoch 동안 오르지 않으면 학습을 멈춥니다(early stopping). 기본 fitness는 mAP50-95라서, 검증 흐름은 그대로 두고 fitness만 대회 지표인 mAP75-95(IoU 0.75–0.95 다섯 단계 AP 평균)로 바꿉니다.

In [ ]:
import time

from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import LOGGER

MAIN_KEY = "metrics/mAP75-95(B)"  # results.csv에 남는 열 이름


def map75_95(box) -> float:
    """검증 결과의 mAP75-95. all_ap는 클래스 × IoU 0.50–0.95(10단계) AP 표라 뒤 5열을 평균한다."""
    return float(box.all_ap[:, 5:].mean()) if len(box.all_ap) else 0.0


class Map7595Trainer(DetectionTrainer):
    """매 epoch 검증의 mAP75-95를 fitness로 쓴다. best.pt와 early stopping이 이 값을 따른다."""

    def validate(self) -> tuple[dict | None, float | None]:
        metrics = self.validator(self)
        if metrics is None:
            return None, None
        metrics.pop("fitness", None)  # 기본 fitness(mAP50-95)는 버린다
        fitness = map75_95(self.validator.metrics.box)
        metrics[MAIN_KEY] = round(fitness, 5)
        if self.best_fitness is None or fitness > self.best_fitness:
            self.best_fitness = fitness
        return metrics, fitness

### 5-2. epoch 진행 표시 (tqdm)

epoch가 끝날 때마다 tqdm 막대를 한 칸 올리고, 그 epoch에 걸린 시간(학습 + 검증)과 검증 지표를 한 줄로 출력합니다. ultralytics 자체 로그는 길어서 학습 중에는 경고만 보이게 줄입니다.

검증 코드가 기울기를 끊는지도 함께 기록합니다. ultralytics 검증 함수는 `torch.inference_mode()` 안에서 돌기 때문에 검증 중에는 `torch.is_grad_enabled()`가 False여야 하고, 학습 중에는 True여야 합니다.

In [ ]:
VAL_KEYS = {
    "P": "metrics/precision(B)",
    "R": "metrics/recall(B)",
    "mAP50": "metrics/mAP50(B)",
    "mAP50-95": "metrics/mAP50-95(B)",
}


class EpochProgress:
    """ultralytics 콜백으로 epoch별 시간·검증 지표·기울기 상태를 모은다."""

    def __init__(self, epochs: int) -> None:
        self.bar = tqdm(total=epochs, desc="학습", unit="epoch")
        self.rows = []  # epoch별 기록
        self.final = None  # 학습이 끝난 뒤 best.pt로 한 검증
        self.epoch_start = None

    def on_train_epoch_start(self, trainer) -> None:
        self.epoch_start = time.time()
        self.grad = {}

    def on_train_batch_end(self, trainer) -> None:
        self.grad.setdefault("train", torch.is_grad_enabled())

    def on_val_start(self, validator) -> None:
        self.val_start = time.time()

    def on_val_batch_end(self, validator) -> None:
        # 검증 배치 안에서 기울기 계산이 꺼져 있는지 본다
        self.grad.setdefault("val", torch.is_grad_enabled())

    def on_val_end(self, validator) -> None:
        self.val_seconds = time.time() - self.val_start

    def on_fit_epoch_end(self, trainer) -> None:
        box = trainer.validator.metrics.box
        metrics = {name: trainer.metrics.get(key) for name, key in VAL_KEYS.items()}
        metrics["mAP75-95"] = map75_95(box)
        # 학습이 끝난 뒤 best.pt 검증에서도 이 콜백이 불린다. epoch 시작 기록이 없으면 그 경우다
        if self.epoch_start is None:
            self.final = metrics
            return
        row = {
            "epoch": trainer.epoch + 1,
            "시간(초)": time.time() - self.epoch_start,
            "검증(초)": self.val_seconds,
            **{k: float(v) for k, v in trainer.label_loss_items(trainer.tloss).items()},
            **metrics,
            "best epoch": trainer.stopper.best_epoch,
            "학습 중 기울기": self.grad.get("train"),
            "검증 중 기울기": self.grad.get("val"),
        }
        self.rows.append(row)
        self.epoch_start = None
        self.bar.update(1)
        self.bar.set_postfix({"mAP75-95": f"{row['mAP75-95']:.4f}", "best": row["best epoch"]})
        tqdm.write(
            f"epoch {row['epoch']:>2} | {row['시간(초)']:6.1f}초 (검증 {row['검증(초)']:.1f}초) | "
            f"P {row['P']:.3f} R {row['R']:.3f} mAP50 {row['mAP50']:.3f} "
            f"mAP75-95 {row['mAP75-95']:.4f} | best epoch {row['best epoch']} | "
            f"기울기 학습 {row['학습 중 기울기']} / 검증 {row['검증 중 기울기']}"
        )

    def callbacks(self) -> dict:
        """model.add_callback에 넘길 {이벤트 이름: 함수}."""
        events = [
            "on_train_epoch_start",
            "on_train_batch_end",
            "on_val_start",
            "on_val_batch_end",
            "on_val_end",
            "on_fit_epoch_end",
        ]
        return {event: getattr(self, event) for event in events}

### 5-3. 학습 실행

설정 셀의 값을 그대로 넘깁니다. `SMOKE = True`면 1 epoch, 학습 사진 10%로 동작만 확인합니다. 같은 실험 이름의 이전 결과는 지우고 새로 학습합니다(ultralytics가 results.csv를 이어 쓰기 때문).

- 전체 파인튜닝: `freeze`를 주지 않습니다.
- 증강: 노션에 정한 값이 없어 ultralytics 기본값을 씁니다.
- 학습 중 검증도 평가와 같은 `conf`, NMS `iou`, `max_det`로 합니다.

In [ ]:
train_args = {
    "data": str(DATA_YAML),
    "epochs": 1 if SMOKE else EPOCHS,
    "fraction": 0.1 if SMOKE else 1.0,
    "patience": PATIENCE,
    "batch": BATCH,
    "imgsz": IMGSZ,
    "optimizer": OPTIMIZER,
    "lr0": LR0,
    "seed": SEED,
    "deterministic": True,
    "device": DEVICE,
    "conf": EVAL_CONF,
    "iou": NMS_IOU,
    "max_det": MAX_DET,
    "project": str(RUNS_OUT_DIR),
    "name": EXP,
    "exist_ok": True,  # 폴더 이름을 EXP로 고정한다 (뒤에 2, 3이 붙지 않게)
}
if EXP_DIR.exists():
    shutil.rmtree(EXP_DIR)

set_seed(SEED)
progress = EpochProgress(train_args["epochs"])
model = YOLO(MODEL)
for event, func in progress.callbacks().items():
    model.add_callback(event, func)

log_level = LOGGER.level
LOGGER.setLevel("WARNING")  # ultralytics 배치별 진행 막대와 정보 로그를 끈다
start = time.time()
try:
    model.train(trainer=Map7595Trainer, **train_args)
finally:
    LOGGER.setLevel(log_level)
    progress.bar.close()
print(f"전체 {(time.time() - start) / 60:.1f}분")

### 5-4. 학습 결과

epoch별 기록과 best.pt 정보, 결과 파일이 저장된 곳입니다. Colab에서는 런타임이 끝나면 이 파일들이 사라지고, 노트북 출력만 남습니다.

In [ ]:
history = pd.DataFrame(progress.rows)
best = history.loc[history["mAP75-95"].idxmax()]
stopped = " (early stopping)" if len(history) < train_args["epochs"] else ""
print(f"학습한 epoch: {len(history)} / {train_args['epochs']}{stopped}")
print(f"epoch당 평균 {history['시간(초)'].mean():.1f}초")
print(f"best epoch {int(best['epoch'])}: mAP75-95 {best['mAP75-95']:.4f}")
if progress.final is not None:
    print("best.pt 최종 검증: " + ", ".join(f"{k} {v:.4f}" for k, v in progress.final.items()))
print(f"best.pt: {WEIGHTS_DIR / 'best.pt'} (있음: {(WEIGHTS_DIR / 'best.pt').exists()})")
history.round(4)

In [ ]:
# 학습 결과가 저장된 곳
show_outputs(EXP_DIR, max_files=20)

## 6. 평가

학습이 끝난 `best.pt`로 val을 다시 평가합니다. 평가 설정은 conf 0.001, NMS IoU 0.5, 최대 탐지 4입니다.

- **mAP75-95**: 주 지표. IoU 0.75–0.95(0.05 간격 5단계) AP의 평균이며 대회 채점 지표와 같습니다.
- **Precision / Recall**: ultralytics 방식으로, 신뢰도 하한을 바꿔 가며 F1(정밀도·재현율의 조화 평균)이 가장 높아지는 지점의 값입니다. conf 0.001에서의 값이 아닙니다.
- **추론속도**: 사진 한 장당 ms. 전처리·추론·후처리(NMS)를 따로 보여 주고 **합계를 대표값**으로 씁니다. `BATCH`장씩 묶어 처리한 시간을 사진 수로 나눈 값이라, 모델끼리는 같은 GPU·같은 배치에서 잰 값끼리만 비교합니다.

In [ ]:
from IPython.display import display

best_pt = WEIGHTS_DIR / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"{best_pt}가 없습니다. 5. 학습을 먼저 실행하세요.")

val_result = YOLO(best_pt).val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=BATCH,
    conf=EVAL_CONF,
    iou=NMS_IOU,
    max_det=MAX_DET,
    device=DEVICE,
    plots=False,
    verbose=False,
    project=str(EXP_DIR),
    name="eval",
    exist_ok=True,
)
box = val_result.box
speed = val_result.speed  # 사진 한 장당 ms. loss는 학습 중 검증에서만 재므로 뺀다
speed_ms = {
    "전처리": speed["preprocess"],
    "추론": speed["inference"],
    "후처리(NMS)": speed["postprocess"],
}
total_ms = sum(speed_ms.values())
gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE

scores = pd.DataFrame(
    [
        {
            "mAP75-95": map75_95(box),
            "mAP50": box.map50,
            "mAP50-95": box.map,
            "Precision": box.mp,
            "Recall": box.mr,
            "추론속도(ms/사진)": total_ms,
        }
    ],
    index=[EXP],
)
speed_table = pd.DataFrame({"ms/사진": {**speed_ms, "합계": total_ms}})

print(f"best.pt: {best_pt}")
print(f"val 사진 {(images['fold'] == FOLD).sum()}장")
print(f"속도 측정: {gpu}, batch {BATCH}, imgsz {IMGSZ}")
display(scores.round(4))
display(speed_table.round(2))

## 7. 혼동행렬 conf 0.001 vs 0.25

`best.pt`로 val을 conf 0.001과 0.25에서 한 번씩 평가해 혼동행렬을 만듭니다. 나머지 설정(NMS IoU 0.5, 최대 탐지 4)은 같습니다. 두 결과를 비교해 혼동행렬에 쓸 conf를 하나로 고정합니다. 0.001 쪽이 칸이 너무 많아 보기 어려우면 혼동행렬만 0.25로 고정합니다.

혼동행렬의 행은 예측, 열은 정답이고 마지막 행·열은 배경입니다. 예측과 정답 박스는 IoU 0.45 이상일 때 짝지어집니다(ultralytics 기준).

- **맞힘**: 정답과 같은 약으로 예측 (대각선)
- **헷갈림**: 박스 위치는 맞았지만 다른 약으로 예측
- **놓침**: 정답 박스에 짝지어진 예측이 없음 (배경 행)
- **오검출**: 정답이 없는 곳에 예측 (배경 열)

In [ ]:
best_model = YOLO(best_pt)
n_classes = len(class_table)
cm = {}
for conf in [EVAL_CONF, PLOT_CONF]:
    result = best_model.val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        conf=conf,
        iou=NMS_IOU,
        max_det=MAX_DET,
        device=DEVICE,
        plots=True,  # 혼동행렬은 plots=True일 때만 계산된다
        verbose=False,
        project=str(EXP_DIR),
        name=f"confusion_conf{conf:g}",
        exist_ok=True,
    )
    cm[conf] = result.confusion_matrix.matrix.copy()  # [예측, 정답], 마지막 행·열은 배경

rows = []
for conf, m in cm.items():
    cls = m[:n_classes, :n_classes]
    rows.append(
        {
            "conf": conf,
            "정답 박스": int(m[:, :n_classes].sum()),
            "맞힘": int(np.trace(cls)),
            "헷갈림": int(cls.sum() - np.trace(cls)),
            "놓침": int(m[n_classes, :n_classes].sum()),
            "오검출": int(m[:n_classes, n_classes].sum()),
            "값이 있는 칸": int((m > 0).sum()),
        }
    )
pd.DataFrame(rows).set_index("conf")

In [ ]:
from matplotlib.colors import LogNorm

# 0인 칸은 흰색으로 두고 값이 있는 칸만 색칠한다. 1–2개인 칸도 보이도록 색은 로그 눈금
# 축 눈금은 원본 약 번호(category_id), bg는 배경
tick_labels = [*class_table["category_id"].astype(str), "bg"]
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
for ax, (conf, m) in zip(axes, cm.items(), strict=True):
    norm = LogNorm(vmin=1, vmax=max(m.max(), 2))
    image = ax.imshow(np.ma.masked_equal(m, 0), cmap="Blues", norm=norm)
    ax.set_title(f"conf {conf:g}  (non-zero cells: {int((m > 0).sum())})")
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_xticks(range(len(tick_labels)), tick_labels, rotation=90, fontsize=5)
    ax.set_yticks(range(len(tick_labels)), tick_labels, fontsize=5)
    ax.grid(False)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
# 다른 약으로 헷갈린 쌍. 개수가 많은 순으로 conf마다 10개
names = class_table.set_index("yolo_idx")[["category_id", "dl_name"]]
pair_tables = []
for conf, m in cm.items():
    cls = m[:n_classes, :n_classes].copy()
    np.fill_diagonal(cls, 0)
    pred_idx, true_idx = np.nonzero(cls)
    pairs = pd.DataFrame(
        {
            "conf": conf,
            "정답 약": names.loc[true_idx, "category_id"].to_numpy(),
            "정답 이름": names.loc[true_idx, "dl_name"].to_numpy(),
            "예측 약": names.loc[pred_idx, "category_id"].to_numpy(),
            "예측 이름": names.loc[pred_idx, "dl_name"].to_numpy(),
            "개수": cls[pred_idx, true_idx].astype(int),
        }
    )
    pair_tables.append(pairs.sort_values("개수", ascending=False).head(10))
pd.concat(pair_tables, ignore_index=True)

## 8. 실패 사례 시각화

학습에 쓰지 않은 val 사진을 `best.pt`로 예측하고, 정답 박스와 예측 박스를 함께 그립니다. 그림에는 신뢰도 `PLOT_CONF`(0.25) 이상인 예측만 그립니다.

예측 박스는 신뢰도가 높은 것부터 아직 짝이 없는 정답 박스 중 가장 많이 겹치는 것과 짝짓고, 아래처럼 나눕니다.

| 종류 | 뜻 | 그림 색 (그림 속 글자) |
|---|---|---|
| 정답 | IoU 0.75 이상, 약도 맞음 | 초록 (ok) |
| 위치 부정확 | 약은 맞지만 IoU 0.5–0.75. 주 지표 mAP75-95에서는 틀린 것과 같다 | 주황 (loose) |
| 다른 약 | 위치는 맞지만(IoU 0.5 이상) 다른 약으로 예측 | 빨강 (wrong) |
| 오검출 | 어떤 정답과도 IoU 0.5 미만 | 보라 (FP) |
| 놓침 | 짝지어진 예측이 없는 정답 박스 | 노랑 굵은 선 (miss) |

정답 박스는 모두 흰 선으로 먼저 그리고, 그 위에 예측 박스를 종류별 색으로 그립니다.

In [ ]:
MATCH_IOU = 0.5  # 이 이상 겹치면 같은 알약을 가리킨 것으로 본다
GOOD_IOU = 0.75  # 주 지표 mAP75-95의 가장 낮은 IoU 기준
KINDS = ["정답", "위치 부정확", "다른 약", "오검출", "놓침"]


def box_iou(box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    """xyxy 박스 하나와 여러 xyxy 박스의 IoU."""
    w = np.clip(np.minimum(box[2], boxes[:, 2]) - np.maximum(box[0], boxes[:, 0]), 0, None)
    h = np.clip(np.minimum(box[3], boxes[:, 3]) - np.maximum(box[1], boxes[:, 1]), 0, None)
    inter = w * h
    area = (box[2] - box[0]) * (box[3] - box[1])
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area + areas - inter)


to_cid = class_table.set_index("yolo_idx")["category_id"]
val_files = images.loc[images["fold"] == FOLD, "file_name"].tolist()
records = []
for result in best_model.predict(
    source=[str(TRAIN_IMG_DIR / f) for f in val_files],
    imgsz=IMGSZ,
    conf=PLOT_CONF,
    iou=NMS_IOU,
    max_det=MAX_DET,
    device=DEVICE,
    stream=True,
    verbose=False,
):
    file_name = Path(result.path).name
    gt = labels[labels["file_name"] == file_name]
    gt_boxes = np.c_[gt["x"], gt["y"], gt["x"] + gt["w"], gt["y"] + gt["h"]].astype(float)
    gt_cids = gt["category_id"].to_numpy()
    matched = np.zeros(len(gt), dtype=bool)
    preds = result.boxes
    for i in preds.conf.argsort(descending=True).tolist():
        box = preds.xyxy[i].cpu().numpy()
        pred_cid = int(to_cid[int(preds.cls[i])])
        ious = np.where(matched, 0.0, box_iou(box, gt_boxes))
        j = int(ious.argmax())
        row = {"file_name": file_name, "pred_cid": pred_cid, "conf": float(preds.conf[i])}
        row |= {"pred_box": box, "iou": float(ious[j])}
        if ious[j] >= MATCH_IOU:
            matched[j] = True
            if pred_cid != gt_cids[j]:
                kind = "다른 약"
            else:
                kind = "위치 부정확" if ious[j] < GOOD_IOU else "정답"
            row |= {"kind": kind, "gt_cid": int(gt_cids[j]), "gt_box": gt_boxes[j]}
        else:
            row |= {"kind": "오검출", "gt_cid": None, "gt_box": None}
        records.append(row)
    for j in np.flatnonzero(~matched):
        records.append(
            {
                "file_name": file_name,
                "kind": "놓침",
                "gt_cid": int(gt_cids[j]),
                "gt_box": gt_boxes[j],
                "pred_cid": None,
                "pred_box": None,
                "iou": np.nan,
                "conf": np.nan,
            }
        )
cases = pd.DataFrame(records)
# 빈 값(None)이 섞여 실수(3351.0)가 되지 않게 정수형으로 둔다
cases[["gt_cid", "pred_cid"]] = cases[["gt_cid", "pred_cid"]].astype("Int64")

# 사진별로 종류마다 몇 개인지 센다
per_image = (
    cases.pivot_table(index="file_name", columns="kind", values="conf", aggfunc="size")
    .reindex(index=val_files, columns=KINDS)
    .fillna(0)
    .astype(int)
)
n_gt = len(labels[labels["file_name"].isin(val_files)])
print(f"val 사진 {len(val_files)}장, 정답 박스 {n_gt}개, 그림용 conf {PLOT_CONF}")
print(f"알약을 놓친 사진: {(per_image['놓침'] > 0).sum()}장")
print(f"잘못 예측한 박스가 있는 사진: {(per_image[KINDS[1:4]].sum(axis=1) > 0).sum()}장")
per_image.sum().to_frame("박스 수").T

In [ ]:
COLORS = {
    "정답": "lime",
    "위치 부정확": "orange",
    "다른 약": "red",
    "오검출": "magenta",
    "놓침": "yellow",
}
SHORT = {"정답": "ok", "위치 부정확": "loose", "다른 약": "wrong", "오검출": "FP", "놓침": "miss"}
N_SHOW = 8  # 종류마다 보여 줄 사진 수


def draw_case(ax: plt.Axes, file_name: str) -> None:
    """사진 한 장에 정답 박스(흰색)와 예측 박스(종류별 색)를 그린다."""
    img = Image.open(TRAIN_IMG_DIR / file_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    gt = labels[labels["file_name"] == file_name]
    for b in gt.itertuples():
        draw.rectangle([b.x, b.y, b.x + b.w, b.y + b.h], outline="white", width=4)
    for c in cases[cases["file_name"] == file_name].itertuples():
        color = COLORS[c.kind]
        if c.kind == "놓침":
            draw.rectangle(list(c.gt_box), outline=color, width=12)
            text = f"miss GT {c.gt_cid}"
            x, y = c.gt_box[:2]
        else:
            draw.rectangle(list(c.pred_box), outline=color, width=6)
            text = f"{SHORT[c.kind]} {c.pred_cid} {c.conf:.2f}"
            if c.kind == "다른 약":
                text += f" (GT {c.gt_cid})"
            elif c.kind == "위치 부정확":
                text += f" IoU {c.iou:.2f}"
            x, y = c.pred_box[:2]
        draw.text((x + 8, y + 8), text, fill=color, font_size=34)
    counts = per_image.loc[file_name]
    ax.imshow(img)
    ax.set_title(
        file_name.split("_")[0]
        + "\n"
        + "  ".join(f"{SHORT[k]} {counts[k]}" for k in KINDS if counts[k]),
        fontsize=9,
    )
    ax.axis("off")


def show_cases(files: list[str], title: str) -> None:
    """사진을 최대 N_SHOW장까지 4열로 그린다."""
    if not files:
        print(f"{title}: 없음")
        return
    files = files[:N_SHOW]
    n_rows = (len(files) + 3) // 4
    fig, axes = plt.subplots(n_rows, 4, figsize=(20, 6.5 * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, file_name in zip(axes.flat, files, strict=False):
        draw_case(ax, file_name)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


# 1) 알약을 놓친 사진: 놓친 수가 많은 순
missed = per_image[per_image["놓침"] > 0].sort_values("놓침", ascending=False)
show_cases(missed.index.tolist(), f"Missed pills ({len(missed)} images)")

In [ ]:
# 2) 잘못 예측한 박스가 있는 사진: 다른 약 → 오검출 → 위치 부정확 순으로 많은 사진부터
wrong_kinds = ["다른 약", "오검출", "위치 부정확"]
wrong = per_image[per_image[wrong_kinds].sum(axis=1) > 0].sort_values(wrong_kinds, ascending=False)
show_cases(wrong.index.tolist(), f"Wrong predictions ({len(wrong)} images)")

In [ ]:
# 실패한 박스 목록 (정답 제외). 약 이름을 붙여 어떤 약에서 틀리는지 본다
name_of = class_table.set_index("category_id")["dl_name"]
failures = cases[cases["kind"] != "정답"].assign(
    combo=lambda df: df["file_name"].str.split("_").str[0],
    gt_name=lambda df: df["gt_cid"].map(name_of),
    pred_name=lambda df: df["pred_cid"].map(name_of),
)
failures["kind"] = pd.Categorical(failures["kind"], categories=KINDS)  # 표 순서를 KINDS 순으로
failures = failures.sort_values(["kind", "file_name"])
cols = ["kind", "combo", "file_name", "gt_cid", "gt_name", "pred_cid", "pred_name", "iou", "conf"]
failures[cols].round(3)